# Jev-Omni マルチモーダル意思決定モデル × AURORA A1 Grand Prix 実践ノートブック (A100 GPU)

Gemma 4 12B-IT をベースとしたオープンウェイトのマルチモーダル意思決定モデル **Jev-Omni (`akhilaaa3/Jev-Omni`)** を Google Colab の **A100 GPU（40GB VRAM）** 上でロードし、公開リポジトリ **[`Sunwood-ai-labs/aurora-a1-grand-prix`](https://github.com/Sunwood-ai-labs/aurora-a1-grand-prix)** をクローンしてリアルタイム AI パイロット推論およびブラウザ走行を行うノートブックです。

### 事前設定
1. メニューの **「ランタイム」 > 「ランタイムのタイプを変更」** を開きます。
2. ハードウェア アクセラレータで **「A100 GPU」** を選択して保存してください。

In [1]:
# GPU デバイスと空き VRAM の確認
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

name, memory.total [MiB], memory.free [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB, 40442 MiB


## 1. 公開リポジトリ (`aurora-a1-grand-prix`) のクローンと依存パッケージの導入
FreeCAD で設計された F1 レースゲーム **[`Sunwood-ai-labs/aurora-a1-grand-prix`](https://github.com/Sunwood-ai-labs/aurora-a1-grand-prix)** をクローンし、Jev-Omni の実行に必要なパッケージをインストールします。

In [2]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q transformers accelerate huggingface_hub pillow soundfile opencv-python-headless
!rm -rf /content/aurora-a1-grand-prix && git clone https://github.com/Sunwood-ai-labs/aurora-a1-grand-prix.git /content/aurora-a1-grand-prix
print("[✓] Cloned https://github.com/Sunwood-ai-labs/aurora-a1-grand-prix.git")

Cloning into '/content/aurora-a1-grand-prix'...
[✓] Cloned https://github.com/Sunwood-ai-labs/aurora-a1-grand-prix.git


## 2. Jev-Omni (`akhilaaa3/Jev-Omni`) の A100 GPU ロード
クローンしたリポジトリの初期化スクリプト（`/content/aurora-a1-grand-prix/tools/colab_init_jev.py` 相当）を用いて、`akhilaaa3/Jev-Omni` を `bfloat16` で A100 GPU（VRAM 使用量 約 23.9 GB）へロードします。

In [3]:
import os
import sys
import time
import json
import torch
from huggingface_hub import snapshot_download

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print(f"PyTorch version : {torch.__version__}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Device      : {torch.cuda.get_device_name(0)} ({vram_gb:.2f} GB)")

t0 = time.time()
model_path = snapshot_download(repo_id="akhilaaa3/Jev-Omni")
if model_path not in sys.path:
    sys.path.insert(0, model_path)

from jev_omni import load_jev_omni
classifier = load_jev_omni()

allocated = torch.cuda.memory_allocated() / 1e9
print(f"[✓] Jev-Omni READY on A100 ({time.time() - t0:.1f}s) | VRAM: {allocated:.2f} GB")

PyTorch version : 2.11.0+cu128
GPU Device      : NVIDIA A100-SXM4-40GB (42.41 GB)
[✓] Jev-Omni READY on A100 (72.4s) | VRAM: 23.92 GB


## 3. 基本テスト：テキスト判定＆マルチモーダル分類
トークン生成を行わず、単一の Forward Pass（約 100 ms）で各選択肢の確率分布を出力することを確認します。

In [4]:
# 1. テキスト二値判定
t0 = time.time()
res1 = classifier.predict(
    state="現在時刻は午前9時00分です。予定されている会議は午前10時00分から開始します。",
    question="会議はすでに始まっていますか？",
    options=["はい", "いいえ"]
)
print(f"[Text 2択] 推論レイテンシ: {(time.time()-t0)*1000:.1f} ms -> {res1['prediction']} ({res1['confidence']*100:.2f}%)")

# 2. 問い合わせルーティング（4択）
t0 = time.time()
res2 = classifier.predict(
    state="問い合わせ: '注文した商品の配送先住所を変更したいのですが、どうすればよいでしょうか？'",
    question="この問い合わせを担当すべき部署はどれですか？",
    options=["技術サポート", "配送・注文管理", "経理・決済窓口", "広報・採用"]
)
print(f"[Text 4択] 推論レイテンシ: {(time.time()-t0)*1000:.1f} ms -> {res2['prediction']} ({res2['confidence']*100:.2f}%)")

[Text 2択] 推論レイテンシ: 102.4 ms -> いいえ (99.97%)
[Text 4択] 推論レイテンシ: 98.1 ms -> 配送・注文管理 (99.63%)


## 4. クローンした `aurora-a1-grand-prix` のゲーム画面で推論テスト
クローンしたリポジトリのコックピット視点画像（`docs/images/cockpit.png`）を、ゲームの `JEV-VISION` モードと同じ質問・選択肢で Jev-Omni に入力します（コース情報は渡さず、画面と速度だけ）。
なお、このゲームの画面から道の向きを当てる精度は低く、選択肢の順番に強く左右されます（README の「結果」を参照）。

In [ ]:
from IPython.display import Image as IPyImage, display

cockpit_img = "/content/aurora-a1-grand-prix/docs/images/cockpit.png"
display(IPyImage(filename=cockpit_img, width=640))

# the same options the game sends in JEV-VISION mode (js/main.js JEV_OPTIONS)
aurora_options = [
    "Drive straight ahead at full throttle",
    "Steer left to follow the left curve",
    "Steer right to follow the right curve",
    "Brake and turn left into the sharp left corner",
    "Brake and turn right into the sharp right corner",
    "Brake hard because the car is about to leave the track",
]

t0 = time.time()
res_race = classifier.predict(
    state="cockpit camera view of your F1 race car on an asphalt circuit. Speed: 180 km/h, gear 5.",
    media=cockpit_img,
    modality="image",
    question="What should the driver do right now to stay on the track and drive fast?",
    options=aurora_options,
)
latency_ms = (time.time() - t0) * 1000
print(f"推論レイテンシ (VISION): {latency_ms:.1f} ms")
print(f"選択アクション: {res_race['prediction']} (確信度: {res_race['confidence']*100:.1f}%)")
print(json.dumps(res_race, indent=2, ensure_ascii=False))

## 5. Colab 内で AURORA A1 Grand Prix を起動してプレイ
クローンした `/content/aurora-a1-grand-prix/tools/jev_bridge.py` のブリッジサーバーをバックグラウンドスレッドで起動し、A100 GPU 上の `classifier` にインプロセスで直結します。
タイトル画面の `PILOT (AI)` で `JEV-VISION`（画面のみ）/ `JEV-TEXT`（コース情報を文章で）を、`AI TIMING` で `REALTIME`（時間を止めない・デフォルト）/ `STEP`（判断を待つ間は時間停止）を選べます。レース中は <kbd>J</kbd> / <kbd>T</kbd> キーで切り替えられます。

In [6]:
import sys
sys.path.insert(0, "/content/aurora-a1-grand-prix/tools")
import jev_bridge

# A100 上の classifier インスタンスを直接バインドしてポート 8765 で起動
server = jev_bridge.start_background_server(port=8765, classifier=classifier)
print("[✓] AURORA A1 Grand Prix + In-Process A100 Jev-Omni server started on port 8765")

try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(8765, height="740")
except Exception:
    print("ブラウザで http://localhost:8765 を開いてください")

[✓] AURORA A1 Grand Prix + In-Process A100 Jev-Omni server started on port 8765
